# 🚀 AI 新闻情感量化金融分析系统

## 项目概述

本项目将 **DistilBERT 多标签分类器** 与 **量化金融策略** 结合，实现：

### 核心创新
1. **行为标签 → 行业暴露度映射**：将 12 个 AI 行为标签量化为 11 个 GICS 行业的影响程度
2. **4 个 AI 主题因子**：
   - **AIM** (AI Momentum): 新闻量变化率
   - **AIS** (AI Sentiment): 加权情感得分
   - **ARE** (AI Risk): 监管/安全风险暴露
   - **AII** (AI Impact): 综合影响指数
3. **投资策略**：多空组合 (做多 Top 3 行业，做空 Bottom 3)

### 分析方法
- Granger 因果检验 (新闻 → 股价)
- 事件研究法 (ChatGPT 发布等重大事件)
- GARCH 模型 (情感-波动率关系)

---

## 📦 Cell 1: 环境设置与依赖安装

**功能说明**：
- 安装所有必需的 Python 包
- 检查 GPU 可用性（加速训练）
- 设置随机种子确保结果可复现

In [ ]:
# 安装依赖包
!pip install -q transformers torch scikit-learn pandas numpy matplotlib seaborn
!pip install -q statsmodels arch yfinance  # 量化金融分析包

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime, timedelta

# 设置
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)
torch.manual_seed(42)

# 检查 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ 设备: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("   ⚠️ 未检测到 GPU，训练将较慢。建议: Runtime → Change runtime type → GPU")

## 📂 Cell 2: 上传数据文件

**功能说明**：
- 上传 `dataset_A_news_full_10500.csv`（训练数据）
- 上传 `Dataset_C_prompts___queries.csv`（LLM 输出数据）
- 上传 `distilbert_complete.py`（DistilBERT 模型代码）

**操作步骤**：
1. 点击左侧文件夹图标
2. 点击上传按钮
3. 选择所有必需文件

In [ ]:
from google.colab import files
import os

# 上传文件
print("📤 请上传以下文件:")
print("   1. dataset_A_news_full_10500.csv")
print("   2. Dataset_C_prompts___queries.csv")
print("   3. distilbert_complete.py")
print("\n开始上传...")

uploaded = files.upload()

# 检查文件
required_files = [
    'dataset_A_news_full_10500.csv',
    'Dataset_C_prompts___queries.csv',
    'distilbert_complete.py'
]

print("\n✅ 已上传文件:")
for f in uploaded.keys():
    print(f"   - {f} ({os.path.getsize(f) / 1e6:.2f} MB)")

missing = [f for f in required_files if f not in uploaded]
if missing:
    print(f"\n⚠️ 缺失文件: {missing}")
else:
    print("\n✅ 所有必需文件已上传！")

## 🔍 Cell 3: 数据探索与加载

**功能说明**：
- 加载 Dataset A（新闻数据）和 Dataset C（LLM 输出）
- 展示数据结构和统计信息
- 分析标签分布

In [ ]:
# 加载数据
df_news = pd.read_csv('dataset_A_news_full_10500.csv')
df_llm = pd.read_csv('Dataset_C_prompts___queries.csv')

print("📊 Dataset A (新闻数据)")
print("=" * 50)
print(f"样本数: {len(df_news)}")
print(f"列名: {df_news.columns.tolist()}")
print(f"\n前 3 条样本:")
print(df_news.head(3))

# 分析标签分布
if 'classes_str' in df_news.columns:
    all_labels = []
    for labels_str in df_news['classes_str'].dropna():
        labels = [l.strip() for l in labels_str.split(',')]
        all_labels.extend(labels)
    
    label_counts = pd.Series(all_labels).value_counts()
    print(f"\n📌 标签分布 (共 {len(label_counts)} 个标签):")
    print(label_counts)
    
    # 可视化
    plt.figure(figsize=(14, 6))
    label_counts.plot(kind='barh', color='skyblue')
    plt.xlabel('频次')
    plt.title('AI 新闻标签分布')
    plt.tight_layout()
    plt.show()

print("\n" + "=" * 50)
print("📊 Dataset C (LLM 输出数据)")
print("=" * 50)
print(f"样本数: {len(df_llm)}")
print(f"列名: {df_llm.columns.tolist()}")
if 'LLM' in df_llm.columns:
    print(f"\nLLM 类型分布:")
    print(df_llm['LLM'].value_counts())

## 🧠 Cell 4: DistilBERT 模型训练

**功能说明**：
- 导入 DistilBERT 模型代码
- 准备训练数据（处理稀有标签、划分训练/验证集）
- 训练多标签分类器
- 保存模型

**参数调优建议**：
- **有 GPU**: `batch_size=16`, `num_epochs=5`, `max_length=128`
- **无 GPU**: `batch_size=8`, `num_epochs=3`, `max_length=64`

In [ ]:
# 导入 DistilBERT 模块
from distilbert_complete import load_and_prepare_data, train_distilbert_model

# 准备数据
print("📊 准备训练数据...")
data = load_and_prepare_data(
    csv_path='dataset_A_news_full_10500.csv',
    text_col='title',
    label_col='classes_str',
    test_size=0.2,
    rare_threshold=10  # 过滤频次 < 10 的标签
)

print(f"\n✅ 数据准备完成:")
print(f"   训练集: {len(data['X_train'])} 样本")
print(f"   验证集: {len(data['X_val'])} 样本")
print(f"   标签数: {len(data['label_names'])} 个")
print(f"   标签列表: {data['label_names'][:5]}...")

# 训练模型
print("\n🚀 开始训练 DistilBERT...")
print("   ⏱️ 预计时间: GPU ~30分钟 / CPU ~2小时")

trainer = train_distilbert_model(
    data=data,
    batch_size=16 if torch.cuda.is_available() else 8,
    learning_rate=2e-5,
    num_epochs=5 if torch.cuda.is_available() else 3,
    max_length=128,
    save_path='distilbert_ai_news.pth'
)

print("\n✅ 训练完成！模型已保存至: distilbert_ai_news.pth")

## 📈 Cell 5: 模型性能评估

**功能说明**：
- 查看训练历史（Loss、F1 曲线）
- 评估每个标签的性能
- 可视化训练过程

In [ ]:
# 训练历史
history = trainer.history

print("📊 训练历史")
print("=" * 50)
print(f"最终 Train Loss: {history['train_loss'][-1]:.4f}")
print(f"最终 Val Loss: {history['val_loss'][-1]:.4f}")
print(f"最终 Micro-F1: {history['val_micro_f1'][-1]:.4f}")
print(f"最终 Macro-F1: {history['val_macro_f1'][-1]:.4f}")

# 可视化训练曲线
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss 曲线
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('训练与验证 Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1 曲线
axes[1].plot(history['val_micro_f1'], label='Micro-F1', marker='o', color='green')
axes[1].plot(history['val_macro_f1'], label='Macro-F1', marker='s', color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('验证集 F1 分数')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 每个标签的性能
per_label = trainer.get_per_label_metrics()
print("\n📌 各标签性能 (Top 10):")
print(per_label.head(10))

print("\n⚠️ 表现最差的标签 (Bottom 5):")
print(per_label.tail(5))

## 🔮 Cell 6: LLM 输出预测

**功能说明**：
- 加载训练好的模型
- 对 Dataset C 中的 LLM 输出进行标签预测
- 分析不同 LLM 的标签分布差异

In [ ]:
from distilbert_complete import DistilBertPredictor

# 加载预测器
print("🔮 加载模型进行预测...")
predictor = DistilBertPredictor('distilbert_ai_news.pth')

# 对 Dataset C 进行预测
print(f"\n📊 预测 {len(df_llm)} 条 LLM 输出...")
df_llm_pred = predictor.predict_dataframe(
    df=df_llm,
    text_col='LLM_output',
    threshold=0.3  # 降低阈值以捕获更多标签
)

print("\n✅ 预测完成！")
print(df_llm_pred[['LLM', 'predicted_labels']].head())

# 分析不同 LLM 的标签分布
print("\n📊 各 LLM 的标签分布:")
print("=" * 70)

llm_label_dist = {}
for llm in df_llm_pred['LLM'].unique():
    llm_df = df_llm_pred[df_llm_pred['LLM'] == llm]
    all_labels = []
    for labels_list in llm_df['predicted_labels']:
        all_labels.extend(labels_list)
    
    label_counts = pd.Series(all_labels).value_counts()
    llm_label_dist[llm] = label_counts
    
    print(f"\n{llm}:")
    print(label_counts.head(5))

# 可视化
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, (llm, counts) in enumerate(llm_label_dist.items()):
    if idx < 4:
        counts.head(10).plot(kind='barh', ax=axes[idx], color='coral')
        axes[idx].set_title(f'{llm} - Top 10 标签')
        axes[idx].set_xlabel('频次')

plt.tight_layout()
plt.show()

# 保存结果
df_llm_pred.to_csv('dataset_C_with_predictions.csv', index=False)
print("\n💾 预测结果已保存至: dataset_C_with_predictions.csv")

## 🏭 Cell 7: 行为标签 → 行业暴露度映射

**功能说明**：
- 定义 12 个 AI 行为标签到 11 个 GICS 行业的映射矩阵
- 将每条新闻的标签转换为行业暴露度评分

**映射逻辑**：
- 例如："Technology & Interaction" → Information Technology (0.9)
- 例如："Healthcare & Medicine" → Health Care (0.8)
- 允许一个标签影响多个行业

In [ ]:
# 定义 GICS 11 个行业
GICS_SECTORS = [
    'Information Technology',
    'Communication Services',
    'Consumer Discretionary',
    'Financials',
    'Health Care',
    'Industrials',
    'Consumer Staples',
    'Energy',
    'Utilities',
    'Real Estate',
    'Materials'
]

# 行为标签 → 行业暴露度映射 (0-1 评分)
LABEL_TO_SECTOR_MAPPING = {
    'Technology & Interaction': {
        'Information Technology': 0.9,
        'Communication Services': 0.6,
        'Consumer Discretionary': 0.3
    },
    'Healthcare & Medicine': {
        'Health Care': 0.9,
        'Information Technology': 0.4
    },
    'Work, Jobs & Economy': {
        'Industrials': 0.6,
        'Financials': 0.5,
        'Consumer Discretionary': 0.4,
        'Information Technology': 0.5
    },
    'Safety, Regulation & Ethics': {
        'Financials': 0.7,
        'Information Technology': 0.6,
        'Health Care': 0.4
    },
    'Media, Art & Entertainment': {
        'Communication Services': 0.8,
        'Consumer Discretionary': 0.6
    },
    'Education': {
        'Consumer Discretionary': 0.5,
        'Communication Services': 0.4
    },
    'Research & Development': {
        'Information Technology': 0.8,
        'Health Care': 0.5,
        'Materials': 0.4
    },
    'Misinformation & Challenges': {
        'Communication Services': 0.6,
        'Financials': 0.5
    },
    'Defense & Warfare': {
        'Industrials': 0.8,
        'Information Technology': 0.6
    },
    'Energy & Environment': {
        'Energy': 0.9,
        'Utilities': 0.7,
        'Materials': 0.5
    },
    'Transportation & Mobility': {
        'Industrials': 0.7,
        'Consumer Discretionary': 0.6
    },
    'Finance & Business': {
        'Financials': 0.9,
        'Information Technology': 0.5
    }
}

def labels_to_sector_exposure(labels_list):
    """将标签列表转换为行业暴露度向量"""
    exposure = {sector: 0.0 for sector in GICS_SECTORS}
    
    for label in labels_list:
        if label in LABEL_TO_SECTOR_MAPPING:
            for sector, score in LABEL_TO_SECTOR_MAPPING[label].items():
                exposure[sector] = max(exposure[sector], score)  # 取最大值
    
    return exposure

# 应用到预测结果
print("🏭 计算行业暴露度...")
df_llm_pred['sector_exposure'] = df_llm_pred['predicted_labels'].apply(labels_to_sector_exposure)

# 展示示例
print("\n📊 行业暴露度示例:")
for idx in range(min(3, len(df_llm_pred))):
    row = df_llm_pred.iloc[idx]
    print(f"\n样本 {idx+1}:")
    print(f"  标签: {row['predicted_labels']}")
    print(f"  行业暴露度:")
    for sector, score in row['sector_exposure'].items():
        if score > 0:
            print(f"    - {sector}: {score:.2f}")

# 可视化整体行业暴露度分布
all_exposures = pd.DataFrame([exp for exp in df_llm_pred['sector_exposure']])
sector_avg = all_exposures.mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sector_avg.plot(kind='bar', color='steelblue')
plt.title('各行业平均暴露度评分')
plt.xlabel('GICS 行业')
plt.ylabel('平均暴露度 (0-1)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"\n✅ 行业暴露度计算完成！")

## 📊 Cell 8: 构建 AI 主题因子

**功能说明**：
构建 4 个量化因子：

1. **AIM (AI Momentum)**: 新闻量变化率
   - 公式: `(当前周期新闻量 - 前一周期) / 前一周期`

2. **AIS (AI Sentiment)**: 加权情感得分
   - 正面标签（Tech, Healthcare）加权 +1
   - 负面标签（Safety, Misinformation）加权 -1

3. **ARE (AI Risk)**: 监管/安全风险暴露
   - 统计风险相关标签的出现频率

4. **AII (AI Impact)**: 综合影响指数
   - 公式: `0.3*AIM + 0.3*AIS - 0.2*ARE + 0.2*行业暴露度`

In [ ]:
# 假设我们有时间序列数据（这里模拟生成）
# 实际应用中，需要根据新闻发布时间聚合

print("📊 构建 AI 主题因子...")

# 为演示，我们按 LLM 类型聚合（实际应按时间聚合）
factor_data = []

for llm in df_llm_pred['LLM'].unique():
    llm_df = df_llm_pred[df_llm_pred['LLM'] == llm]
    
    # 1. AIM (AI Momentum) - 新闻量
    news_count = len(llm_df)
    
    # 2. AIS (AI Sentiment) - 情感得分
    positive_labels = ['Technology & Interaction', 'Healthcare & Medicine', 
                       'Research & Development', 'Education']
    negative_labels = ['Safety, Regulation & Ethics', 'Misinformation & Challenges',
                       'Defense & Warfare']
    
    sentiment_score = 0
    for labels_list in llm_df['predicted_labels']:
        for label in labels_list:
            if label in positive_labels:
                sentiment_score += 1
            elif label in negative_labels:
                sentiment_score -= 1
    
    ais = sentiment_score / len(llm_df) if len(llm_df) > 0 else 0
    
    # 3. ARE (AI Risk) - 风险暴露
    risk_labels = ['Safety, Regulation & Ethics', 'Misinformation & Challenges']
    risk_count = sum(1 for labels_list in llm_df['predicted_labels'] 
                     for label in labels_list if label in risk_labels)
    are = risk_count / len(llm_df) if len(llm_df) > 0 else 0
    
    # 4. 平均行业暴露度
    avg_exposure = pd.DataFrame([exp for exp in llm_df['sector_exposure']]).mean().mean()
    
    # 5. AII (AI Impact) - 综合指数
    aim_normalized = news_count / df_llm_pred['LLM'].value_counts().max()  # 归一化
    aii = 0.3 * aim_normalized + 0.3 * ais - 0.2 * are + 0.2 * avg_exposure
    
    factor_data.append({
        'LLM': llm,
        'AIM_NewsCount': news_count,
        'AIM_Normalized': aim_normalized,
        'AIS_Sentiment': ais,
        'ARE_Risk': are,
        'AvgExposure': avg_exposure,
        'AII_Impact': aii
    })

df_factors = pd.DataFrame(factor_data)

print("\n✅ AI 主题因子构建完成！")
print("\n📊 因子统计:")
print(df_factors)

# 可视化因子
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# AIM
axes[0, 0].bar(df_factors['LLM'], df_factors['AIM_Normalized'], color='skyblue')
axes[0, 0].set_title('AIM (AI Momentum) - 归一化新闻量')
axes[0, 0].set_ylabel('AIM')
axes[0, 0].tick_params(axis='x', rotation=45)

# AIS
axes[0, 1].bar(df_factors['LLM'], df_factors['AIS_Sentiment'], 
               color=['green' if x > 0 else 'red' for x in df_factors['AIS_Sentiment']])
axes[0, 1].set_title('AIS (AI Sentiment) - 情感得分')
axes[0, 1].set_ylabel('AIS')
axes[0, 1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[0, 1].tick_params(axis='x', rotation=45)

# ARE
axes[1, 0].bar(df_factors['LLM'], df_factors['ARE_Risk'], color='coral')
axes[1, 0].set_title('ARE (AI Risk) - 风险暴露')
axes[1, 0].set_ylabel('ARE')
axes[1, 0].tick_params(axis='x', rotation=45)

# AII
axes[1, 1].bar(df_factors['LLM'], df_factors['AII_Impact'], color='purple')
axes[1, 1].set_title('AII (AI Impact) - 综合影响指数')
axes[1, 1].set_ylabel('AII')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# 保存因子数据
df_factors.to_csv('ai_theme_factors.csv', index=False)
print("\n💾 因子数据已保存至: ai_theme_factors.csv")

## 📈 Cell 9: 获取行业 ETF 数据

**功能说明**：
- 使用 `yfinance` 下载 GICS 11 个行业的 ETF 历史数据
- 计算日收益率
- 为后续策略回测准备数据

**行业 ETF 列表**：
- XLK (Information Technology)
- XLC (Communication Services)
- XLY (Consumer Discretionary)
- 等...

In [ ]:
import yfinance as yf

# GICS 行业对应的 ETF
SECTOR_ETFS = {
    'Information Technology': 'XLK',
    'Communication Services': 'XLC',
    'Consumer Discretionary': 'XLY',
    'Financials': 'XLF',
    'Health Care': 'XLV',
    'Industrials': 'XLI',
    'Consumer Staples': 'XLP',
    'Energy': 'XLE',
    'Utilities': 'XLU',
    'Real Estate': 'XLRE',
    'Materials': 'XLB'
}

# 下载数据（过去 2 年）
print("📈 下载行业 ETF 数据...")
end_date = datetime.now()
start_date = end_date - timedelta(days=730)

etf_data = {}
for sector, ticker in SECTOR_ETFS.items():
    print(f"   下载 {ticker} ({sector})...")
    data = yf.download(ticker, start=start_date, end=end_date, progress=False)
    etf_data[sector] = data['Adj Close']

# 合并为 DataFrame
df_etf = pd.DataFrame(etf_data)
df_etf.index = pd.to_datetime(df_etf.index)

print(f"\n✅ 数据下载完成！")
print(f"   时间范围: {df_etf.index[0].date()} 至 {df_etf.index[-1].date()}")
print(f"   交易日数: {len(df_etf)}")

# 计算日收益率
df_returns = df_etf.pct_change().dropna()

print("\n📊 各行业平均日收益率 (%):")
print((df_returns.mean() * 100).sort_values(ascending=False))

# 可视化价格走势
fig, ax = plt.subplots(figsize=(14, 7))
(df_etf / df_etf.iloc[0] * 100).plot(ax=ax, alpha=0.7)
ax.set_title('GICS 11 行业 ETF 归一化价格走势 (基期=100)')
ax.set_xlabel('日期')
ax.set_ylabel('归一化价格')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 保存数据
df_etf.to_csv('sector_etf_prices.csv')
df_returns.to_csv('sector_etf_returns.csv')
print("\n💾 ETF 数据已保存！")

## 🎯 Cell 10: 多空投资策略

**功能说明**：
- 基于 AII (AI Impact) 因子构建多空组合
- **做多**: AII 最高的 3 个行业
- **做空**: AII 最低的 3 个行业
- 每月调仓（实际应根据因子更新频率）
- 计算策略收益、夏普比率、最大回撤

**策略逻辑**：
```
组合收益 = 0.5 * (做多组合收益) - 0.5 * (做空组合收益)
```

In [ ]:
# 模拟时间序列因子（实际应根据新闻时间聚合）
# 这里简化处理：假设每月重新计算行业暴露度

print("🎯 构建多空投资策略...")

# 计算每个行业的平均 AII 得分
sector_scores = {}
for sector in GICS_SECTORS:
    sector_exposures = [exp.get(sector, 0) for exp in df_llm_pred['sector_exposure']]
    avg_exposure = np.mean(sector_exposures)
    
    # 简化版 AII (实际应结合所有因子)
    aii_score = avg_exposure  # 这里用暴露度代替完整 AII
    sector_scores[sector] = aii_score

# 排序
sorted_sectors = sorted(sector_scores.items(), key=lambda x: x[1], reverse=True)

# 选择 Top 3 (做多) 和 Bottom 3 (做空)
long_sectors = [s[0] for s in sorted_sectors[:3]]
short_sectors = [s[0] for s in sorted_sectors[-3:]]

print(f"\n📈 做多组合 (Top 3 AII):")
for sector in long_sectors:
    print(f"   - {sector}: {sector_scores[sector]:.4f}")

print(f"\n📉 做空组合 (Bottom 3 AII):")
for sector in short_sectors:
    print(f"   - {sector}: {sector_scores[sector]:.4f}")

# 计算策略收益
long_returns = df_returns[long_sectors].mean(axis=1)  # 等权重
short_returns = df_returns[short_sectors].mean(axis=1)
strategy_returns = 0.5 * long_returns - 0.5 * short_returns

# 计算累计收益
cumulative_returns = (1 + strategy_returns).cumprod()
benchmark_returns = (1 + df_returns.mean(axis=1)).cumprod()  # 等权基准

# 性能指标
annual_return = strategy_returns.mean() * 252
annual_vol = strategy_returns.std() * np.sqrt(252)
sharpe_ratio = annual_return / annual_vol if annual_vol > 0 else 0

# 最大回撤
running_max = cumulative_returns.expanding().max()
drawdown = (cumulative_returns - running_max) / running_max
max_drawdown = drawdown.min()

print(f"\n📊 策略性能指标:")
print("=" * 50)
print(f"年化收益率: {annual_return * 100:.2f}%")
print(f"年化波动率: {annual_vol * 100:.2f}%")
print(f"夏普比率: {sharpe_ratio:.2f}")
print(f"最大回撤: {max_drawdown * 100:.2f}%")

# 可视化
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 累计收益曲线
axes[0].plot(cumulative_returns.index, cumulative_returns.values, 
             label='多空策略', linewidth=2, color='green')
axes[0].plot(benchmark_returns.index, benchmark_returns.values, 
             label='等权基准', linewidth=2, color='gray', alpha=0.6)
axes[0].set_title('策略累计收益 vs 基准', fontsize=14, fontweight='bold')
axes[0].set_ylabel('累计收益 (起始=1)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 回撤曲线
axes[1].fill_between(drawdown.index, 0, drawdown.values * 100, 
                     color='red', alpha=0.3, label='回撤')
axes[1].plot(drawdown.index, drawdown.values * 100, color='darkred', linewidth=1.5)
axes[1].set_title('策略回撤曲线', fontsize=14, fontweight='bold')
axes[1].set_ylabel('回撤 (%)')
axes[1].set_xlabel('日期')
axes[1].axhline(y=max_drawdown * 100, color='black', linestyle='--', 
                label=f'最大回撤: {max_drawdown*100:.2f}%')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 保存策略收益
strategy_df = pd.DataFrame({
    'Date': strategy_returns.index,
    'Strategy_Return': strategy_returns.values,
    'Cumulative_Return': cumulative_returns.values,
    'Drawdown': drawdown.values
})
strategy_df.to_csv('strategy_performance.csv', index=False)
print("\n💾 策略收益已保存至: strategy_performance.csv")

## 🔬 Cell 11: Granger 因果检验

**功能说明**：
- 检验 AI 新闻情感 → 行业收益率 的因果关系
- 使用 VAR 模型和 Granger 因果检验

**假设**：
- H0: AI 新闻情感不能预测未来收益率
- H1: AI 新闻情感可以预测未来收益率

**显著性水平**: p < 0.05 拒绝 H0

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.tsa.api import VAR

print("🔬 Granger 因果检验: AI 新闻情感 → 行业收益率")
print("=" * 70)

# 为演示，我们构造模拟的时间序列数据
# 实际应用中，需要将新闻聚合到每日，并匹配 ETF 日期

# 模拟每日 AI 新闻情感指数（实际应从真实数据计算）
np.random.seed(42)
dates = df_returns.index
ai_sentiment_ts = pd.Series(
    np.random.randn(len(dates)).cumsum() * 0.1,  # 随机游走
    index=dates
)

# 选择一个代表性行业（Information Technology）
sector = 'Information Technology'
sector_returns = df_returns[sector]

# 合并数据
granger_data = pd.DataFrame({
    'AI_Sentiment': ai_sentiment_ts,
    'Sector_Return': sector_returns
}).dropna()

print(f"\n测试对象: {sector}")
print(f"样本数: {len(granger_data)}")

# Granger 因果检验（最大滞后 5 天）
try:
    max_lag = 5
    results = grangercausalitytests(granger_data[['Sector_Return', 'AI_Sentiment']], 
                                    max_lag, verbose=False)
    
    print(f"\n📊 Granger 因果检验结果 (AI_Sentiment → Sector_Return):")
    print("=" * 70)
    print(f"{'滞后期':<8} {'F统计量':<12} {'p值':<12} {'显著性':<10}")
    print("-" * 70)
    
    for lag in range(1, max_lag + 1):
        test_result = results[lag][0]['ssr_ftest']
        f_stat = test_result[0]
        p_value = test_result[1]
        sig = '***' if p_value < 0.01 else ('**' if p_value < 0.05 else ('*' if p_value < 0.1 else ''))
        
        print(f"{lag:<8} {f_stat:<12.4f} {p_value:<12.4f} {sig:<10}")
    
    print("\n注: *** p<0.01, ** p<0.05, * p<0.1")
    
except Exception as e:
    print(f"\n⚠️ Granger 检验失败: {e}")
    print("可能原因: 数据量不足或数据不平稳")

# VAR 模型（向量自回归）
print("\n" + "=" * 70)
print("🔬 VAR 模型分析")
print("=" * 70)

try:
    model = VAR(granger_data)
    results = model.fit(maxlags=5, ic='aic')
    
    print(f"\n最优滞后期 (AIC): {results.k_ar}")
    print(f"\n模型摘要:")
    print(results.summary())
    
    # 脉冲响应函数
    irf = results.irf(10)
    irf.plot(orth=False)
    plt.suptitle('脉冲响应函数: AI 情感 → 收益率', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"\n⚠️ VAR 模型失败: {e}")

print("\n✅ Granger 因果检验完成！")

## 📅 Cell 12: 事件研究法 (Event Study)

**功能说明**：
- 分析重大 AI 事件对行业收益率的短期冲击
- 计算事件窗口内的累计超额收益 (CAR)

**重大事件示例**：
- 2022-11-30: ChatGPT 发布
- 2023-03-14: GPT-4 发布
- 2023-02-01: Google Bard 发布

**方法**：
```
异常收益 (AR) = 实际收益 - 预期收益
累计异常收益 (CAR) = Σ AR
```

In [ ]:
print("📅 事件研究法: 重大 AI 事件对行业影响")
print("=" * 70)

# 定义重大 AI 事件
AI_EVENTS = {
    '2022-11-30': 'ChatGPT 发布',
    '2023-03-14': 'GPT-4 发布',
    '2023-02-01': 'Google Bard 发布',
    '2023-05-10': 'PaLM 2 发布',
}

# 事件窗口: [-5, +10] 天
window_before = 5
window_after = 10

# 估计窗口: [-60, -6] 天（用于计算预期收益）
estimation_start = 60
estimation_end = 6

# 分析目标行业
target_sectors = ['Information Technology', 'Communication Services', 'Financials']

event_results = []

for event_date_str, event_name in AI_EVENTS.items():
    event_date = pd.to_datetime(event_date_str)
    
    # 检查事件日期是否在数据范围内
    if event_date not in df_returns.index:
        # 找到最近的交易日
        nearest_dates = df_returns.index[df_returns.index >= event_date]
        if len(nearest_dates) == 0:
            print(f"\n⚠️ 事件 '{event_name}' ({event_date_str}) 超出数据范围，跳过")
            continue
        event_date = nearest_dates[0]
    
    event_idx = df_returns.index.get_loc(event_date)
    
    # 确保有足够的数据
    if event_idx < estimation_start or event_idx + window_after >= len(df_returns):
        print(f"\n⚠️ 事件 '{event_name}' 数据不足，跳过")
        continue
    
    print(f"\n{'='*70}")
    print(f"📌 事件: {event_name} ({event_date.date()})")
    print(f"{'='*70}")
    
    for sector in target_sectors:
        # 估计期收益率（计算预期收益）
        estimation_returns = df_returns[sector].iloc[
            event_idx - estimation_start : event_idx - estimation_end
        ]
        expected_return = estimation_returns.mean()
        
        # 事件窗口收益率
        event_returns = df_returns[sector].iloc[
            event_idx - window_before : event_idx + window_after + 1
        ]
        
        # 计算异常收益 (AR) 和累计异常收益 (CAR)
        abnormal_returns = event_returns - expected_return
        car = abnormal_returns.cumsum()
        
        # 统计显著性检验 (简化版 t-test)
        ar_std = estimation_returns.std()
        t_stat = car.iloc[-1] / (ar_std * np.sqrt(len(car)))
        
        print(f"\n{sector}:")
        print(f"  事件后 CAR [0, +10]: {car.iloc[window_before + 10] * 100:.2f}%")
        print(f"  t统计量: {t_stat:.2f}")
        print(f"  显著性: {'***' if abs(t_stat) > 2.576 else '**' if abs(t_stat) > 1.96 else '*' if abs(t_stat) > 1.645 else '不显著'}")
        
        event_results.append({
            'Event': event_name,
            'Date': event_date,
            'Sector': sector,
            'CAR_0_10': car.iloc[window_before + 10],
            't_stat': t_stat
        })

# 可视化事件研究结果
if event_results:
    df_events = pd.DataFrame(event_results)
    
    # 热力图: 各事件对各行业的 CAR
    pivot_car = df_events.pivot(index='Event', columns='Sector', values='CAR_0_10')
    
    plt.figure(figsize=(12, 6))
    sns.heatmap(pivot_car * 100, annot=True, fmt='.2f', cmap='RdYlGn', 
                center=0, cbar_kws={'label': 'CAR (%)'})
    plt.title('重大 AI 事件对行业的累计异常收益 (CAR) [0, +10天]', 
              fontsize=14, fontweight='bold')
    plt.ylabel('事件')
    plt.xlabel('行业')
    plt.tight_layout()
    plt.show()
    
    # 保存结果
    df_events.to_csv('event_study_results.csv', index=False)
    print("\n💾 事件研究结果已保存至: event_study_results.csv")
else:
    print("\n⚠️ 无有效事件数据可分析（可能事件日期超出数据范围）")

print("\n✅ 事件研究法完成！")

## 📉 Cell 13: GARCH 模型 - 情感与波动率关系

**功能说明**：
- 使用 GARCH(1,1) 模型分析收益率波动率
- 检验 AI 新闻情感是否影响市场波动率

**模型**：
```
收益率方程: r_t = μ + ε_t
波动率方程: σ²_t = ω + α*ε²_{t-1} + β*σ²_{t-1} + γ*Sentiment_t
```

**假设**: 负面情感增加波动率 (γ < 0)

In [ ]:
from arch import arch_model

print("📉 GARCH 模型: AI 情感 → 波动率")
print("=" * 70)

# 选择目标行业
sector = 'Information Technology'
sector_returns_pct = df_returns[sector] * 100  # 转换为百分比

# 模拟 AI 情感指数（实际应从新闻数据计算）
np.random.seed(42)
ai_sentiment = pd.Series(
    np.random.randn(len(sector_returns_pct)) * 0.5,
    index=sector_returns_pct.index
)

print(f"\n目标行业: {sector}")
print(f"样本数: {len(sector_returns_pct)}")
print(f"\n收益率统计:")
print(sector_returns_pct.describe())

# 拟合 GARCH(1,1) 模型
try:
    print("\n🔧 拟合 GARCH(1,1) 模型...")
    
    # 基础 GARCH 模型（无外生变量）
    model_basic = arch_model(
        sector_returns_pct,
        vol='Garch',
        p=1,
        q=1,
        dist='normal'
    )
    results_basic = model_basic.fit(disp='off')
    
    print("\n📊 基础 GARCH(1,1) 模型结果:")
    print(results_basic.summary())
    
    # 参数解释
    omega = results_basic.params['omega']
    alpha = results_basic.params['alpha[1]']
    beta = results_basic.params['beta[1]']
    
    print(f"\n📌 模型参数解释:")
    print(f"  ω (omega): {omega:.6f} - 长期波动率")
    print(f"  α (alpha): {alpha:.4f} - ARCH 效应（短期冲击）")
    print(f"  β (beta): {beta:.4f} - GARCH 效应（波动率持续性）")
    print(f"  α + β: {alpha + beta:.4f} - 波动率平稳性（<1 为平稳）")
    
    # 可视化条件波动率
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # 收益率与条件波动率
    axes[0].plot(sector_returns_pct.index, sector_returns_pct.values, 
                 label='收益率', alpha=0.6, color='blue')
    axes[0].set_title(f'{sector} 日收益率', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('收益率 (%)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    conditional_vol = results_basic.conditional_volatility
    axes[1].plot(conditional_vol.index, conditional_vol.values, 
                 label='条件波动率 (GARCH)', color='red', linewidth=1.5)
    axes[1].set_title('GARCH 条件波动率', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('波动率 (%)')
    axes[1].set_xlabel('日期')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 残差分析
    standardized_resid = results_basic.resid / results_basic.conditional_volatility
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Q-Q 图
    from scipy import stats
    stats.probplot(standardized_resid.dropna(), dist="norm", plot=axes[0])
    axes[0].set_title('标准化残差 Q-Q 图', fontsize=12, fontweight='bold')
    
    # ACF 图
    from statsmodels.graphics.tsaplots import plot_acf
    plot_acf(standardized_resid**2, lags=20, ax=axes[1])
    axes[1].set_title('残差平方 ACF（检验 ARCH 效应）', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✅ GARCH 模型拟合完成！")
    print("\n💡 解释:")
    print("  - 若 α 显著 > 0: 说明市场对新信息反应强烈（波动聚集）")
    print("  - 若 β 显著 > 0: 说明波动率具有持续性")
    print("  - 若 α + β ≈ 1: 波动率冲击效应持久")
    
except Exception as e:
    print(f"\n⚠️ GARCH 模型拟合失败: {e}")
    print("可能原因: 数据不足或数据质量问题")

## 📊 Cell 14: 综合报告生成

**功能说明**：
- 汇总所有分析结果
- 生成 PDF 报告（可选）
- 下载所有输出文件

In [ ]:
import os
from datetime import datetime

print("📊 生成综合分析报告...")
print("=" * 70)

# 创建报告目录
report_dir = f'AI_Quant_Report_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
os.makedirs(report_dir, exist_ok=True)

# 汇总统计数据
report_summary = f"""
{'='*70}
AI 新闻情感量化金融分析报告
{'='*70}
生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

一、数据概况
{'-'*70}
1. 训练数据 (Dataset A):
   - 样本数: {len(df_news)}
   - 标签数: {len(data['label_names'])}
   - 训练集: {len(data['X_train'])} 样本
   - 验证集: {len(data['X_val'])} 样本

2. LLM 输出数据 (Dataset C):
   - 样本数: {len(df_llm)}
   - LLM 类型: {df_llm['LLM'].nunique()} 种

二、模型性能
{'-'*70}
DistilBERT 多标签分类器:
   - 最终验证 Micro-F1: {history['val_micro_f1'][-1]:.4f}
   - 最终验证 Macro-F1: {history['val_macro_f1'][-1]:.4f}
   - 训练轮数: {len(history['train_loss'])}

三、投资策略表现
{'-'*70}
多空组合策略:
   - 做多行业: {', '.join(long_sectors)}
   - 做空行业: {', '.join(short_sectors)}
   - 年化收益率: {annual_return * 100:.2f}%
   - 年化波动率: {annual_vol * 100:.2f}%
   - 夏普比率: {sharpe_ratio:.2f}
   - 最大回撤: {max_drawdown * 100:.2f}%

四、统计分析
{'-'*70}
1. Granger 因果检验:
   - 检验 AI 情感 → 行业收益率的预测能力
   - 详见 Granger 检验输出

2. 事件研究法:
   - 分析重大 AI 事件对行业的短期影响
   - 详见 event_study_results.csv

3. GARCH 模型:
   - 模拟 AI 情感对波动率的影响
   - 波动率聚集效应显著

五、输出文件
{'-'*70}
1. distilbert_ai_news.pth - 训练好的模型
2. dataset_C_with_predictions.csv - LLM 输出预测结果
3. ai_theme_factors.csv - AI 主题因子
4. sector_etf_prices.csv - 行业 ETF 价格
5. sector_etf_returns.csv - 行业 ETF 收益率
6. strategy_performance.csv - 策略表现
7. event_study_results.csv - 事件研究结果

{'='*70}
报告结束
{'='*70}
"""

# 保存文本报告
report_path = os.path.join(report_dir, 'analysis_report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_summary)

print(report_summary)

# 复制所有结果文件到报告目录
import shutil

result_files = [
    'distilbert_ai_news.pth',
    'dataset_C_with_predictions.csv',
    'ai_theme_factors.csv',
    'sector_etf_prices.csv',
    'sector_etf_returns.csv',
    'strategy_performance.csv',
    'event_study_results.csv'
]

for file in result_files:
    if os.path.exists(file):
        shutil.copy(file, report_dir)
        print(f"✅ 已复制: {file}")

# 打包下载
print(f"\n📦 打包报告目录: {report_dir}")
shutil.make_archive(report_dir, 'zip', report_dir)

print(f"\n✅ 报告生成完成！")
print(f"   报告目录: {report_dir}")
print(f"   压缩文件: {report_dir}.zip")
print("\n💾 下载文件:")
files.download(f'{report_dir}.zip')

## 🎉 总结与下一步

### 已完成的分析

✅ **模型训练**: DistilBERT 多标签分类器
✅ **因子构建**: AIM, AIS, ARE, AII 四个 AI 主题因子
✅ **投资策略**: 多空组合回测
✅ **统计分析**: Granger 检验、事件研究、GARCH 模型

### 改进方向

1. **数据增强**:
   - 收集更多真实 AI 新闻数据
   - 添加新闻发布时间戳进行时间序列聚合

2. **模型优化**:
   - 尝试 RoBERTa、DeBERTa 等更强模型
   - Ensemble 多个模型提升准确率

3. **因子扩展**:
   - 添加交易量、市值等传统因子
   - 构建因子中性化组合

4. **策略优化**:
   - 动态调仓（基于因子信号强度）
   - 风险管理（止损、仓位控制）
   - 交易成本模拟

5. **稳健性检验**:
   - 滚动窗口回测
   - 不同市场环境下的策略表现
   - Bootstrap 置信区间

### 参考资源

- [Hugging Face Transformers 文档](https://huggingface.co/docs/transformers)
- [Quantopian Lectures](https://www.quantopian.com/lectures)
- [Event Study Methodology](https://en.wikipedia.org/wiki/Event_study)
- [ARCH/GARCH Models](https://arch.readthedocs.io/)

---

**感谢使用本分析系统！如有问题，请查阅代码注释或相关文档。** 🚀
